In [3]:
!pip install rioxarray geopandas tqdm
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis


In [5]:
# ==========================================================
# Empirical Quantile Mapping (EQM) Bias Correction
# PRECIPITATION ONLY
# For Multiple CMIP6 Models and Multiple Scenarios
#
# Equations applied (per Themebl et al., 2012; Gudmundsson, 2014):
#   P_his(d)* = F_obs,m^{-1} [ F_his,m ( P_his,m ) ]   -- Eq. 1
#   P_sim(d)* = F_obs,m^{-1} [ F_sim,m ( P_sim,m ) ]   -- Eq. 2
# ==========================================================

import os
import gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ==========================================================
# USER SETTINGS
# ==========================================================

INPUT_ROOT  = "Stationwise/Precipitation/Raw_GCM_Precipitation"
OUTPUT_ROOT = "Stationwise/Precipitation/BiasCorrected_QM_Precipitation"

OBS_FILE  = "Stationwise/Karnali_1986_2015.xlsx"
OBS_SHEET = "Precipitation"

SCENARIOS = ["historical", "ssp245", "ssp585"]

CAL_START = "1986-01-01"
CAL_END   = "2014-12-31"

# Number of quantile bins
# 99  → 1st–99th percentile table (as described in the paper)
# 199 or 999 → finer resolution for better extreme value correction
N_QUANTILES = 99

# ==========================================================
# HELPER – BUILD MONTHLY EMPIRICAL CDFs
# ==========================================================

def build_monthly_cdfs(series: pd.Series, n_quantiles: int = 99) -> dict:
    """
    Build one empirical CDF (quantile table) per calendar month.

    Parameters
    ----------
    series      : daily time series with a DatetimeIndex
    n_quantiles : number of percentile points

    Returns
    -------
    dict { month (1-12): {"probs": array, "values": array} }
    """
    probs = np.linspace(0, 1, n_quantiles + 2)   # 0.0 … 1.0 inclusive
    cdfs  = {}
    for m in range(1, 13):
        vals = series[series.index.month == m].dropna().values
        if len(vals) == 0:
            cdfs[m] = {"probs": probs, "values": np.full_like(probs, np.nan)}
        else:
            cdfs[m] = {"probs": probs, "values": np.quantile(vals, probs)}
    return cdfs


# ==========================================================
# HELPER – APPLY EQM
# ==========================================================

def apply_eqm(raw_series: pd.Series,
              cdf_raw: dict,
              cdf_obs: dict) -> pd.Series:
    """
    Apply Empirical Quantile Mapping month by month.

    For each daily value x:
      1. Find its quantile rank p in the raw monthly CDF  → F_raw,m(x)
      2. Map p through the inverse observed monthly CDF   → F_obs,m^{-1}(p)

    Parameters
    ----------
    raw_series : daily GCM precipitation values (DatetimeIndex)
    cdf_raw    : monthly CDFs of the raw GCM series
    cdf_obs    : monthly CDFs of the observed series

    Returns
    -------
    pd.Series : bias-corrected daily precipitation, same index as raw_series
    """
    corrected = raw_series.copy().astype(float)

    for m in range(1, 13):
        mask = raw_series.index.month == m
        if not mask.any():
            continue

        raw_vals = raw_series.loc[mask].values
        r_probs  = cdf_raw[m]["probs"]
        r_vals   = cdf_raw[m]["values"]
        o_probs  = cdf_obs[m]["probs"]
        o_vals   = cdf_obs[m]["values"]

        if np.all(np.isnan(r_vals)) or np.all(np.isnan(o_vals)):
            continue

        # Step 1 – forward CDF: get quantile rank of each raw value
        p_ranks = np.interp(raw_vals, r_vals, r_probs)

        # Step 2 – inverse CDF: map rank through observed distribution
        corrected.loc[mask] = np.interp(p_ranks, o_probs, o_vals)

    return corrected


# ==========================================================
# READ OBSERVED PRECIPITATION
# ==========================================================

print("=" * 70)
print("Reading Observed Precipitation ...")
print("=" * 70)

df_obs = pd.read_excel(OBS_FILE, sheet_name=OBS_SHEET)

for col in ["year", "month", "days"]:
    if col in df_obs.columns:
        df_obs.drop(columns=col, inplace=True)

df_obs["Date"] = pd.to_datetime(df_obs["Date"])
df_obs.set_index("Date", inplace=True)
df_obs.index = df_obs.index.normalize()
df_obs.columns = df_obs.columns.astype(str)

print(f"Observed stations : {len(df_obs.columns)}")
print(f"Observed days     : {len(df_obs)}")

# ==========================================================
# FIND AVAILABLE MODELS
# ==========================================================

models = sorted([
    d for d in os.listdir(INPUT_ROOT)
    if os.path.isdir(os.path.join(INPUT_ROOT, d))
])

print(f"\nModels found ({len(models)}):")
for m in models:
    print("  ", m)

# ==========================================================
# MAIN LOOP
# ==========================================================

summary = []

for model in tqdm(models, desc="Models", colour="green"):
    print(f"\n{'='*70}")
    print(f"MODEL : {model}")
    print(f"{'='*70}")

    # ------------------------------------------------------
    # Load Historical GCM Precipitation
    # ------------------------------------------------------
    hist_file = os.path.join(
        INPUT_ROOT, model, "historical",
        f"{model}_historical_karnali_pr.csv"
    )

    if not os.path.exists(hist_file):
        print("  Historical precipitation file missing – skipping model.")
        continue

    df_hist = pd.read_csv(hist_file)
    df_hist["Date"] = pd.to_datetime(df_hist["Date"])
    df_hist.set_index("Date", inplace=True)
    df_hist.index = df_hist.index.normalize()
    df_hist.columns = df_hist.columns.astype(str)

    # ------------------------------------------------------
    # Common dates & calibration slice
    # ------------------------------------------------------
    common_dates = df_obs.index.intersection(df_hist.index)
    obs_common   = df_obs.loc[common_dates]
    hist_common  = df_hist.loc[common_dates]

    print(f"  Common dates     : {len(common_dates)}")

    obs_cal  = obs_common.loc[CAL_START:CAL_END]
    hist_cal = hist_common.loc[CAL_START:CAL_END]

    print(f"  Calibration days : {len(obs_cal)}")

    # ------------------------------------------------------
    # Automatic station matching  (obs_id → gcm_col)
    # ------------------------------------------------------
    station_map = {}
    for obs_st in df_obs.columns:
        for gcm_col in df_hist.columns:
            if gcm_col.startswith(obs_st):
                station_map[obs_st] = gcm_col
                break

    print(f"  Matched stations : {len(station_map)}")
    if not station_map:
        print("  No matched stations – skipping model.")
        continue

    # ------------------------------------------------------
    # Build calibration-period CDFs
    #   obs_cdfs  : observed  → used as F_obs,m^{-1}  in all equations
    #   hist_cdfs : historical GCM → used as F_his,m   in Eq. 1
    # ------------------------------------------------------
    obs_cdfs  = {}
    hist_cdfs = {}

    for obs_st in tqdm(station_map, desc="Building CDFs", leave=False, colour="yellow"):
        gcm_col = station_map[obs_st]
        obs_cdfs[obs_st]  = build_monthly_cdfs(obs_cal[obs_st],   N_QUANTILES)
        hist_cdfs[obs_st] = build_monthly_cdfs(hist_cal[gcm_col], N_QUANTILES)

    print("  Monthly CDFs built.")

    # ======================================================
    # APPLY CORRECTION SCENARIO BY SCENARIO
    # ======================================================
    for scenario in tqdm(SCENARIOS, desc=f"{model}", leave=False, colour="blue"):

        input_file = os.path.join(
            INPUT_ROOT, model, scenario,
            f"{model}_{scenario}_karnali_pr.csv"
        )

        if not os.path.exists(input_file):
            print(f"\n  [{scenario}] file not found – skipping.")
            continue

        print(f"\n  Processing {scenario} …")

        try:
            df = pd.read_csv(input_file)
            df["Date"] = pd.to_datetime(df["Date"])
            df.set_index("Date", inplace=True)
            df.index = df.index.normalize()
            df.columns = df.columns.astype(str)

            corrected = df.copy()

            # Choose which raw CDF to use for the forward step:
            #   historical → Eq. 1 : F_his,m  (calibration-period GCM CDF)
            #   future     → Eq. 2 : F_sim,m  (full future scenario CDF)
            if scenario == "historical":
                raw_cdfs = hist_cdfs
            else:
                raw_cdfs = {}
                for obs_st in tqdm(station_map,
                                   desc="Future CDFs",
                                   leave=False,
                                   colour="cyan"):
                    gcm_col = station_map[obs_st]
                    if gcm_col in df.columns:
                        raw_cdfs[obs_st] = build_monthly_cdfs(
                            df[gcm_col], N_QUANTILES
                        )

            # Apply EQM station by station
            for obs_st in tqdm(station_map,
                               desc=f"EQM {scenario}",
                               leave=False,
                               colour="cyan"):
                gcm_col = station_map[obs_st]
                if gcm_col not in corrected.columns:
                    continue
                if obs_st not in raw_cdfs:
                    continue

                corrected[gcm_col] = apply_eqm(
                    raw_series = df[gcm_col],
                    cdf_raw    = raw_cdfs[obs_st],
                    cdf_obs    = obs_cdfs[obs_st]
                )

            # Precipitation cannot be negative – clip to zero
            for gcm_col in station_map.values():
                if gcm_col in corrected.columns:
                    corrected[gcm_col] = corrected[gcm_col].clip(lower=0)

            # --------------------------------------------------
            # Save output
            # --------------------------------------------------
            out_dir = os.path.join(OUTPUT_ROOT, model, scenario)
            os.makedirs(out_dir, exist_ok=True)

            output_file = os.path.join(
                out_dir,
                f"{model}_{scenario}_pr_qm_bias_corrected.csv"
            )
            corrected.to_csv(output_file)
            print(f"  Saved : {output_file}")

            summary.append({
                "Model":    model,
                "Scenario": scenario,
                "Stations": len(station_map),
                "Days":     len(corrected),
                "Output":   output_file,
                "Status":   "Success"
            })

        except Exception as e:
            print(f"  Error processing {scenario}: {e}")
            summary.append({
                "Model":    model,
                "Scenario": scenario,
                "Stations": 0,
                "Days":     0,
                "Output":   "",
                "Status":   f"Failed: {e}"
            })

        finally:
            if "df"        in locals(): del df
            if "corrected" in locals(): del corrected
            gc.collect()

# ==========================================================
# SUMMARY REPORT
# ==========================================================

df_summary = pd.DataFrame(summary)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
summary_path = os.path.join(OUTPUT_ROOT, "qm_precipitation_summary.csv")
df_summary.to_csv(summary_path, index=False)

print("\n" + "=" * 70)
print("EQM PRECIPITATION BIAS CORRECTION — COMPLETE")
print("=" * 70)
print(df_summary.to_string(index=False))
print(f"\nSummary saved to : {summary_path}")

Reading Observed Precipitation ...
Observed stations : 10
Observed days     : 10957

Models found (1):
   ACCESS-CM2


Models:   0%|          | 0/1 [00:00<?, ?it/s]


MODEL : ACCESS-CM2
  Common dates     : 10592
  Calibration days : 10592
  Matched stations : 10


Building CDFs:   0%|          | 0/10 [00:00<?, ?it/s]

  Monthly CDFs built.


ACCESS-CM2:   0%|          | 0/3 [00:00<?, ?it/s]


  Processing historical …


EQM historical:   0%|          | 0/10 [00:00<?, ?it/s]

  Saved : Stationwise/Precipitation/BiasCorrected_QM_Precipitation/ACCESS-CM2/historical/ACCESS-CM2_historical_pr_qm_bias_corrected.csv

  [ssp245] file not found – skipping.

  [ssp585] file not found – skipping.

EQM PRECIPITATION BIAS CORRECTION — COMPLETE
     Model   Scenario  Stations  Days                                                                                                                        Output  Status
ACCESS-CM2 historical        10 10957 Stationwise/Precipitation/BiasCorrected_QM_Precipitation/ACCESS-CM2/historical/ACCESS-CM2_historical_pr_qm_bias_corrected.csv Success

Summary saved to : Stationwise/Precipitation/BiasCorrected_QM_Precipitation/qm_precipitation_summary.csv
